# ML-10 — Week 7: Content Action Playbook (Lane 4 — CTR / Engagement Opportunity Scoring)

Week 5 produced a scored queue. Week 6 established what that score is and is not allowed to claim. Neither of them is something a person can act on: a column of probabilities tells a reviewer nothing about *what to do on Monday*, *why this page*, or *when to stop trusting the list*.

This notebook turns the validated output into the thing that actually ships — **a ranked queue with reason codes, an archetype → action map, the rules a human must apply before acting, an explicit no-go list, and the monitoring that says when the whole thing has gone stale.**

The model is rebuilt from scratch here (same data, same label, same client-grouped split, same seed) so every number below is produced by this run, not quoted from last week. Section 5 exports the queue and figures the paper builds on next week.

> Working with an AI assistant? Tell it to read `skills/README.md` first, then load `writing-honest-claims` + `flyrank/flyrank-data`.

**Careful words throughout:** *observed / measured / directional / decision-support*. Nothing here claims a rewrite causes clicks — no such design was run. The strongest honest sentence in this notebook is *"these pages look worth opening first, and here is what a reviewer should check."*

**Non-production by design.** This is a reviewer's worksheet, not a service. There is no API, no scheduler, no write-back to any client system, and nothing in this notebook takes an action on its own.

In [1]:
# ---- Setup: rebuild the Week-5 scored universe, in this run -------------------------------
import json, sys, textwrap
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display

SEED = 42
np.random.seed(SEED)

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / CSV).exists()), None)
if root is None:
    root = Path.cwd()
    df = pd.read_csv("https://raw.githubusercontent.com/nothaziq/FlyRank-ML-Week1/main/" + CSV)
else:
    df = pd.read_csv(root / CSV)

OUT = root / "work" / "outputs"
FIG = root / "work" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | seed {SEED}")
print(f"{len(df):,} rows x {df.shape[1]} columns  (rates such as ctr are x100 percentages)")
print(f"exports -> {OUT.relative_to(root)}/  and  {FIG.relative_to(root)}/")

python 3.12.10 | pandas 3.0.6 | scikit-learn 1.9.1 | seed 42
30,000 rows x 44 columns  (rates such as ctr are x100 percentages)
exports -> work\outputs/  and  work\figures/


In [2]:
# ---- The Week-5 design, unchanged: earlier window -> later window ---------------------------
BANDS = [0, 3, 5, 7, 10, 15, 20]
BAND_LABELS = ["0-3", "3-5", "5-7", "7-10", "10-15", "15-20"]
IMP_FLOOR_30 = 500 / 3          # Week-4's 90d floor of 500 impressions, on a 30d window
MIN_EXP_CLICKS_30 = 10 / 3      # Week-4's 10 expected clicks, on a 30d window
SHORTFALL = 0.5                 # "less than half the band norm"

d = df.copy()
d["band"] = pd.cut(d.avg_position, BANDS, labels=BAND_LABELS)
d["ctr_prev"] = d.clicks_prev_30d / d.impressions_prev_30d.replace(0, np.nan) * 100
d["ctr_last"] = d.clicks_last_30d / d.impressions_last_30d.replace(0, np.nan) * 100

vis = d.band.notna()
elig_prev = vis & (d.impressions_prev_30d >= IMP_FLOOR_30)
elig_last = vis & (d.impressions_last_30d >= IMP_FLOOR_30)

d["norm_prev"] = d.ctr_prev.where(elig_prev).groupby(d.band, observed=True).transform("median")
d["norm_last"] = d.ctr_last.where(elig_last).groupby(d.band, observed=True).transform("median")
d["exp_clicks_prev"] = d.impressions_prev_30d * d.norm_prev / 100
d["exp_clicks_last"] = d.impressions_last_30d * d.norm_last / 100

# Label: the shortfall is STILL there one window later (a defined-rule proxy, not a reviewed outcome)
d["persisted_shortfall"] = ((d.exp_clicks_last >= MIN_EXP_CLICKS_30) &
                            (d.ctr_last < SHORTFALL * d.norm_last)).astype(int)
# Week-4 rule, run on the earlier window only
d["rule_flagged"] = elig_prev & (d.exp_clicks_prev >= MIN_EXP_CLICKS_30) & (d.ctr_prev < SHORTFALL * d.norm_prev)
d["baseline_score"] = np.where(d.rule_flagged, d.exp_clicks_prev - d.clicks_prev_30d, 0.0)

lane = d[elig_prev & elig_last].copy().reset_index(drop=True)
DROPPED_LATER = int((elig_prev & ~elig_last).sum())
BASE_RATE = float(lane.persisted_shortfall.mean())

lane["log_impressions_prev"] = np.log1p(lane.impressions_prev_30d)
lane["ctr_vs_band_norm"] = lane.ctr_prev / lane.norm_prev
lane["shortfall_clicks_prev"] = lane.exp_clicks_prev - lane.clicks_prev_30d
lane["sessions_per_click_prev"] = lane.sessions_prev_30d / lane.clicks_prev_30d.replace(0, np.nan)
lane["log_word_count"] = np.log1p(lane.word_count)
lane["log_age_days"] = np.log1p(lane.content_age_days)
lane["log_search_volume"] = np.log1p(lane.search_volume)

NUM = ["log_impressions_prev", "ctr_prev", "ctr_vs_band_norm", "shortfall_clicks_prev",
       "sessions_per_click_prev", "avg_position", "log_word_count", "log_age_days",
       "days_since_last_update", "log_search_volume", "competition", "cpc"]
CAT = ["band", "content_type", "main_intent", "freshness_tier"]
FEATURES = NUM + CAT

X = lane[FEATURES].copy()
X[NUM] = X[NUM].replace([np.inf, -np.inf], np.nan)
y = lane.persisted_shortfall.to_numpy()
groups = lane.client_id.to_numpy()

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), CAT),
])
model = Pipeline([("pre", pre), ("m", RandomForestClassifier(
    n_estimators=400, min_samples_leaf=5, class_weight="balanced_subsample",
    random_state=SEED, n_jobs=-1))])

folds = list(GroupKFold(n_splits=5).split(X, y, groups))
oof = np.zeros(len(lane))
fold_p50, fold_p50_rule = [], []

def precision_at_k(score, truth, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")[:k]
    return float(np.mean(np.asarray(truth)[order]))

for tr, te in folds:
    model.fit(X.iloc[tr], y[tr])
    oof[te] = model.predict_proba(X.iloc[te])[:, 1]
    fold_p50.append(precision_at_k(oof[te], y[te], 50))
    fold_p50_rule.append(precision_at_k(lane.baseline_score.to_numpy()[te], y[te], 50))

lane["p_model"] = oof
P50_MODEL = precision_at_k(oof, y, 50)
P50_RULE = precision_at_k(lane.baseline_score.to_numpy(), y, 50)
P50_FOLD_MEAN, P50_FOLD_SD = float(np.mean(fold_p50)), float(np.std(fold_p50))

print(f"Universe: {len(lane):,} pages, {lane.client_id.nunique()} clients, "
      f"base rate {BASE_RATE:.4f}, dropped in later window {DROPPED_LATER:,}")
print(f"Out-of-fold P@50 -- model {P50_MODEL:.3f} | Week-4 rule {P50_RULE:.3f} | random order {BASE_RATE:.3f}")
print(f"Per-fold P@50    -- model {P50_FOLD_MEAN:.3f} (sd {P50_FOLD_SD:.3f}) | rule {np.mean(fold_p50_rule):.3f}")
print("\nReminder from Week 6: the pooled model-vs-rule gap sits inside that fold-to-fold spread.")
print("The playbook below is built on 'not worse than the rule, plausibly somewhat better' -- not on a win.")

Universe: 9,836 pages, 28 clients, base rate 0.0952, dropped in later window 1,683
Out-of-fold P@50 -- model 0.920 | Week-4 rule 0.880 | random order 0.095
Per-fold P@50    -- model 0.648 (sd 0.151) | rule 0.624

Reminder from Week 6: the pooled model-vs-rule gap sits inside that fold-to-fold spread.
The playbook below is built on 'not worse than the rule, plausibly somewhat better' -- not on a win.


In [3]:
# ---- Reproduction check against the committed Week-5 receipts ------------------------------
w5_path = OUT / "w05_model_metrics.json"
if w5_path.exists():
    w5 = json.loads(w5_path.read_text())
    checks = [
        ("universe pages", len(lane), w5["universe"]["pages"]),
        ("clients", lane.client_id.nunique(), w5["universe"]["clients"]),
        ("dropped in later window", DROPPED_LATER, w5["universe"]["dropped_not_eligible_later_window"]),
        ("base rate", round(BASE_RATE, 4), w5["universe"]["base_rate"]),
        ("rule P@50", round(P50_RULE, 3), w5["results"]["Rule baseline (Week 4)"]["P@50"]),
        ("model P@50", round(P50_MODEL, 3), w5["results"]["Random Forest"]["P@50"]),
    ]
    rep = pd.DataFrame(checks, columns=["quantity", "this run", "Week 5 (committed)"])
    rep["match"] = np.where(rep["this run"] == rep["Week 5 (committed)"], "yes", "DIFFERS")
    rep["this run"] = rep["this run"].map(lambda v: f"{v:,}" if isinstance(v, int) else f"{v}")
    rep["Week 5 (committed)"] = rep["Week 5 (committed)"].map(lambda v: f"{v:,}" if isinstance(v, int) else f"{v}")
    display(rep)
    print(f"Week 5 ran on scikit-learn {w5['design']['sklearn']}; this run is {sklearn.__version__}.")
    print("Universe, clients and base rate are library-independent and must match exactly.")
    print("Precision@K can move a point or two across library versions -- if it does, this run's numbers are the ones cited below.")
else:
    print("w05_model_metrics.json not found -- this notebook is self-contained and uses this run's numbers.")

,quantity,this run,Week 5 (committed),match
0,universe pages,9836.0,9836.0,yes
1,clients,28.0,28.0,yes
2,dropped in later window,1683.0,1683.0,yes
3,base rate,0.0952,0.0952,yes
4,rule P@50,0.88,0.88,yes
5,model P@50,0.92,0.92,yes


Week 5 ran on scikit-learn 1.9.1; this run is 1.9.1.
Universe, clients and base rate are library-independent and must match exactly.
Precision@K can move a point or two across library versions -- if it does, this run's numbers are the ones cited below.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

A probability is not an action. Three things have to be attached to it before a reviewer can use the list:

1. **A tier** — how much evidence is behind this row, expressed as an observed hit rate rather than a number between 0 and 1.
2. **An archetype** — which *kind* of page this is, from transparent rules on inputs the reviewer can see, which is what determines the suggested action.
3. **Reason codes** — the specific measured facts that put this page on the list, so a reviewer can disagree with the machine on the evidence rather than on vibes.

**The ordering is the model's; the action is the archetype's.** That separation matters. The model is good at *which pages first* (that is what precision@K measured). It was never trained on, and has no evidence about, *what fixes a page* — so the action is never learned, it is a lookup from a rule a human wrote and can change without retraining anything.

**Tiers, and what they are allowed to promise.** Tier thresholds are cut on the out-of-fold probability and then labelled with the **observed** rate of persisted shortfall in each tier, so the tier name carries a measured number and not an adjective:

| Tier | Out-of-fold probability | What the reviewer is told |
|---|---|---|
| **A — review first** | p ≥ 0.60 | roughly half of these had a shortfall that was still there a month later |
| **B — review if slots remain** | 0.30 ≤ p < 0.60 | about one in four |
| **C — monitor only, do not work** | p < 0.30 | near the base rate; a reviewer slot here is close to random |

Tier C exists so the list has a floor. The most common failure mode for a ranked queue is that somebody works it to the bottom.

In [4]:
# ---- Reason codes: the measured facts that put a page on the list --------------------------
# Every code is computed from the EARLIER window or static metadata only -- the same information
# a reviewer would have on the Monday they open the queue. The evidence numbers live in their own
# columns so the codes stay groupable and the reviewer can check the machine's arithmetic.
q = lane.copy()
q["ctr_ratio"] = q.ctr_vs_band_norm                       # CTR as a multiple of its position band's norm
q["click_gap_30d"] = q.shortfall_clicks_prev.clip(lower=0)  # expected-minus-actual clicks, earlier window
q["sessions_per_click"] = q.sessions_per_click_prev

THIN_WORDS = float(q.word_count.quantile(0.25))
HIGH_IMPRESSIONS = 2000
LOW_IMPRESSIONS = 400
SPC_LO, SPC_HI = 0.5, 2.0     # GA4 sessions per GSC click; far from ~1 means the two systems disagree
STALE_DAYS = 90

def reason_codes(r):
    codes = []
    if r.ctr_ratio < 0.5:
        codes.append("ctr_far_below_band_norm")
    elif r.ctr_ratio < 0.8:
        codes.append("ctr_below_band_norm")
    if r.click_gap_30d >= 10:
        codes.append("large_absolute_click_gap")
    if r.impressions_prev_30d >= HIGH_IMPRESSIONS:
        codes.append("high_impression_volume")
    if r.impressions_prev_30d < LOW_IMPRESSIONS:
        codes.append("low_volume_noisy_signal")
    if r.band in ("0-3", "3-5"):
        codes.append("page_one_position")
    elif r.band in ("5-7", "7-10"):
        codes.append("page_two_position")
    else:
        codes.append("deep_position")
    if r.word_count < THIN_WORDS and r.main_intent == "informational":
        codes.append("thin_for_informational_intent")
    if r.main_intent in ("transactional", "commercial") and r.content_type == "keyword article":
        codes.append("commercial_intent_generic_format")
    if r.main_intent == "navigational":
        codes.append("navigational_intent_norms_do_not_apply")
    if pd.isna(r.sessions_per_click) or not (SPC_LO <= r.sessions_per_click <= SPC_HI):
        codes.append("analytics_click_session_mismatch")
    if r.days_since_last_update >= STALE_DAYS:
        codes.append("no_recent_update")
    if (r.p_model >= 0.60) and (not r.rule_flagged):
        codes.append("model_rule_disagreement")
    return codes

q["reason_codes"] = [reason_codes(r) for r in q.itertuples()]

freq = Counter(c for codes in q.reason_codes for c in codes)
print(f"Reason-code frequency across the full scored universe (n={len(q):,} pages):")
display(pd.DataFrame(sorted(freq.items(), key=lambda kv: -kv[1]), columns=["reason_code", "pages"])
        .assign(share=lambda t: (t.pages / len(q)).round(3)))
print("A code that fires on almost everything carries almost no information. Two to read carefully:")
print(f"  analytics_click_session_mismatch fires on {freq['analytics_click_session_mismatch']/len(q):.0%} of pages --")
print("    this is the GA4-vs-GSC gap Week 4 found. It is a data-quality flag, NOT a content finding.")
print(f"  commercial_intent_generic_format fires on {freq['commercial_intent_generic_format']/len(q):.0%} --")
print(f"    {q.content_type.value_counts(normalize=True).iloc[0]:.0%} of this slice is one content_type, so the code is")
print("    mostly describing the portfolio, not the page. It is kept as context, never as a ranking reason.")

Reason-code frequency across the full scored universe (n=9,836 pages):


,reason_code,pages,share
0,analytics_click_session_mismatch,4918,0.500
1,page_two_position,4530,0.461
2,high_impression_volume,3973,0.404
3,commercial_intent_generic_format,3927,0.399
4,no_recent_update,3580,0.364
5,deep_position,3517,0.358
6,ctr_far_below_band_norm,2915,0.296
7,page_one_position,1789,0.182
8,low_volume_noisy_signal,1158,0.118
9,ctr_below_band_norm,1045,0.106


A code that fires on almost everything carries almost no information. Two to read carefully:
  analytics_click_session_mismatch fires on 50% of pages --
    this is the GA4-vs-GSC gap Week 4 found. It is a data-quality flag, NOT a content finding.
  commercial_intent_generic_format fires on 40% --
    99% of this slice is one content_type, so the code is
    mostly describing the portfolio, not the page. It is kept as context, never as a ranking reason.


In [5]:
# ---- Tiers: cut on probability, labelled with the OBSERVED rate ----------------------------
TIER_EDGES = [-0.001, 0.30, 0.60, 1.001]
TIER_NAMES = ["C - monitor only", "B - review if slots remain", "A - review first"]
q["tier"] = pd.cut(q.p_model, TIER_EDGES, labels=TIER_NAMES)

tiers = (q.groupby("tier", observed=True)
         .agg(pages=("persisted_shortfall", "size"),
              observed_shortfall_rate=("persisted_shortfall", "mean"),
              median_click_gap_30d=("click_gap_30d", "median"),
              clients=("client_id", "nunique"))
         .reindex(TIER_NAMES[::-1]))
tiers["lift_vs_base"] = tiers.observed_shortfall_rate / BASE_RATE
print(f"Tiers, scored out-of-fold. Base rate for comparison: {BASE_RATE:.3f}")
display(tiers.round(3))
print("Read: the tier label is a measured hit rate on this data, not a confidence adjective.")
print("Tier C is ~2% -- a slot spent there is close to a coin weighted against you. It is on the list")
print("so that 'work the queue until you run out of time' cannot happen by accident.")

Tiers, scored out-of-fold. Base rate for comparison: 0.095


,pages,observed_shortfall_rate,median_click_gap_30d,clients,lift_vs_base
tier,,,,,
A - review first,912,0.530,7.630,19,5.565
B - review if slots remain,1275,0.225,1.968,20,2.365
C - monitor only,7649,0.022,0.000,28,0.228


Read: the tier label is a measured hit rate on this data, not a confidence adjective.
Tier C is ~2% -- a slot spent there is close to a coin weighted against you. It is on the list
so that 'work the queue until you run out of time' cannot happen by accident.


### Archetype → action

Seven archetypes, assigned by **first match in priority order** so every page gets exactly one, from inputs a reviewer can see on the page itself. The action attached to each is a *review instruction*, not a fix — the reviewer decides what the page needs.

| Archetype | Assigned when | Suggested first action | Why this action, honestly |
|---|---|---|---|
| `page_one_snippet_gap` | CTR < 0.5× band norm, position 0–5 | review title + meta description against the query intent | the page already ranks where clicks are available; what the searcher sees in the SERP is the part that has not been tested |
| `page_two_climber` | CTR < 0.5× band norm, position 5–10 | review title/meta **and** on-page relevance | at these positions both the snippet and the match to the query plausibly limit clicks; the data cannot separate the two |
| `deep_page_shortfall` | CTR < 0.5× band norm, position 10–20 | relevance and internal-linking review; **low priority** | CTR this deep is noisy and position is the bigger lever; a snippet rewrite here is optimising the wrong variable |
| `thin_informational` | word count in the bottom quartile, informational intent | review whether the page actually answers the query | measured as *short for its intent* — shortness is not a defect by itself |
| `commercial_intent_review` | transactional/commercial intent | human intent check **before** any content work | the page may be the wrong *format* for the query; a machine cannot judge that |
| `navigational_no_action` | navigational intent | **no action** — remove from queue | band norms are built on non-branded behaviour and do not describe branded queries |
| `no_clear_archetype` | none of the above | manual triage | the honest label when the rules do not fit; better than forcing a page into a bucket |

**The archetype does not affect the ranking.** It is attached after the sort. So a reviewer can throw out an entire archetype — or FlyRank can rewrite one row of this table — without touching the model.

In [6]:
# ---- Archetype assignment: first match wins ------------------------------------------------
def archetype(r):
    if r.main_intent == "navigational":
        return "navigational_no_action"
    if r.ctr_ratio < 0.5 and r.band in ("0-3", "3-5"):
        return "page_one_snippet_gap"
    if r.ctr_ratio < 0.5 and r.band in ("5-7", "7-10"):
        return "page_two_climber"
    if r.ctr_ratio < 0.5 and r.band in ("10-15", "15-20"):
        return "deep_page_shortfall"
    if r.word_count < THIN_WORDS and r.main_intent == "informational":
        return "thin_informational"
    if r.main_intent in ("transactional", "commercial"):
        return "commercial_intent_review"
    return "no_clear_archetype"

ACTION = {
    "page_one_snippet_gap":   ("review_title_and_meta", "high"),
    "page_two_climber":       ("review_title_meta_and_onpage_relevance", "high"),
    "deep_page_shortfall":    ("relevance_and_internal_link_review", "low"),
    "thin_informational":     ("check_page_answers_the_query", "medium"),
    "commercial_intent_review": ("human_intent_check_before_content_work", "medium"),
    "navigational_no_action": ("no_action_remove_from_queue", "none"),
    "no_clear_archetype":     ("manual_triage", "low"),
}

q["archetype"] = [archetype(r) for r in q.itertuples()]
q["suggested_action"] = q.archetype.map(lambda a: ACTION[a][0])
q["action_priority"] = q.archetype.map(lambda a: ACTION[a][1])

QUEUE_K = 200   # a working quarter for one reviewer at ~50 slots a week
q = q.sort_values("p_model", ascending=False).reset_index(drop=True)
q["queue_rank"] = np.arange(1, len(q) + 1)
q["in_top_200"] = q.queue_rank <= QUEUE_K
top = q[q.in_top_200]

mix = (pd.DataFrame({"universe": q.archetype.value_counts(), "top_200": top.archetype.value_counts()})
       .fillna(0).astype(int))
mix["top_200_hit_rate"] = top.groupby("archetype").persisted_shortfall.mean().round(3)
mix["share_of_universe"] = (mix.universe / len(q)).round(3)
print(f"Archetype mix. Hit rate = observed share whose shortfall persisted; base rate {BASE_RATE:.3f}.")
display(mix.sort_values("top_200", ascending=False))
print("Honest reading of this table:")
print("  - The queue is dominated by one archetype. That is a finding about the portfolio, not a bug:")
print("    the model's top pages are overwhelmingly high-volume page-2 pages with a large click gap.")
print("  - Hit rates on rows with fewer than ~30 pages are not to be quoted -- n is printed next to them")
print("    for exactly that reason (writing-honest-claims: report n or drop the ratio).")
nca = q[q.archetype == "no_clear_archetype"]
print(f"  - {len(nca)/len(q):.0%} of the universe falls through to no_clear_archetype. That is a large share and it")
print(f"    is left as-is rather than forced into a bucket: {(nca.tier == 'C - monitor only').mean():.0%} of those pages are tier C")
print(f"    (median queue rank {nca.queue_rank.median():,.0f} of {len(q):,}), so the fallthrough sits where nobody works.")

Archetype mix. Hit rate = observed share whose shortfall persisted; base rate 0.095.


,universe,top_200,top_200_hit_rate,share_of_universe
archetype,,,,
page_two_climber,1266,160,0.850,0.129
page_one_snippet_gap,396,31,0.677,0.040
deep_page_shortfall,1251,5,0.800,0.127
no_clear_archetype,3424,4,0.750,0.348
commercial_intent_review,2823,0,NaN,0.287
navigational_no_action,8,0,NaN,0.001
thin_informational,668,0,NaN,0.068


Honest reading of this table:
  - The queue is dominated by one archetype. That is a finding about the portfolio, not a bug:
    the model's top pages are overwhelmingly high-volume page-2 pages with a large click gap.
  - Hit rates on rows with fewer than ~30 pages are not to be quoted -- n is printed next to them
    for exactly that reason (writing-honest-claims: report n or drop the ratio).
  - 35% of the universe falls through to no_clear_archetype. That is a large share and it
    is left as-is rather than forced into a bucket: 84% of those pages are tier C
    (median queue rank 5,136 of 9,836), so the fallthrough sits where nobody works.


In [7]:
# ---- What the reviewer actually opens: the top of the queue --------------------------------
# client_id is an anonymized hash in the source file; it is relabelled to an ordinal here so nothing
# in the notebook output can be traced to a client, and the code lists are truncated for reading.
client_rank = {c: f"client {i+1:02d}" for i, c in enumerate(q.client_id.value_counts().index)}
q["client_label"] = q.client_id.map(client_rank)
top = q[q.in_top_200]

preview = (q.head(12)[["queue_rank", "tier", "client_label", "band", "impressions_prev_30d",
                       "ctr_prev", "norm_prev", "ctr_ratio", "click_gap_30d",
                       "archetype", "suggested_action", "p_model", "reason_codes"]]
           .assign(reason_codes=lambda t: t.reason_codes.map(lambda cs: ", ".join(cs[:3]) + ("  +…" if len(cs) > 3 else ""))))
print("Top of the ranked queue (12 of "
      f"{len(q):,} scored pages). ctr_prev and norm_prev are %CTR; ctr_ratio is their quotient.")
display(preview.round({"ctr_prev": 2, "norm_prev": 2, "ctr_ratio": 2, "click_gap_30d": 1, "p_model": 3}))
print(f"Top {QUEUE_K}: {top.client_label.nunique()} clients represented, "
      f"largest holds {top.client_label.value_counts().iloc[0] / QUEUE_K:.0%} "
      f"(that client is {q.client_label.value_counts().iloc[0] / len(q):.0%} of the universe).")
print("That concentration is a standing review rule, not a footnote -- see Section 3.")

Top of the ranked queue (12 of 9,836 scored pages). ctr_prev and norm_prev are %CTR; ctr_ratio is their quotient.


,queue_rank,tier,client_label,band,impressions_prev_30d,ctr_prev,norm_prev,ctr_ratio,click_gap_30d,archetype,suggested_action,p_model,reason_codes
0,1,A - review first,client 01,3-5,32546,0.05,0.32,0.14,88.9,page_one_snippet_gap,review_title_and_meta,0.958,"ctr_far_below_band_norm, large_absolute_click_..."
1,2,A - review first,client 01,5-7,8177,0.02,0.23,0.11,16.7,page_two_climber,review_title_meta_and_onpage_relevance,0.958,"ctr_far_below_band_norm, large_absolute_click_..."
2,3,A - review first,client 02,5-7,9055,0.01,0.23,0.05,19.7,page_two_climber,review_title_meta_and_onpage_relevance,0.956,"ctr_far_below_band_norm, large_absolute_click_..."
3,4,A - review first,client 02,5-7,14416,0.03,0.23,0.12,28.9,page_two_climber,review_title_meta_and_onpage_relevance,0.951,"ctr_far_below_band_norm, large_absolute_click_..."
4,5,A - review first,client 02,5-7,5396,0.02,0.23,0.08,11.3,page_two_climber,review_title_meta_and_onpage_relevance,0.947,"ctr_far_below_band_norm, large_absolute_click_..."
5,6,A - review first,client 01,5-7,23941,0.06,0.23,0.27,39.7,page_two_climber,review_title_meta_and_onpage_relevance,0.946,"ctr_far_below_band_norm, large_absolute_click_..."
6,7,A - review first,client 02,5-7,9947,0.09,0.23,0.40,13.7,page_two_climber,review_title_meta_and_onpage_relevance,0.944,"ctr_far_below_band_norm, large_absolute_click_..."
7,8,A - review first,client 02,5-7,22650,0.01,0.23,0.04,49.7,page_two_climber,review_title_meta_and_onpage_relevance,0.944,"ctr_far_below_band_norm, large_absolute_click_..."
8,9,A - review first,client 01,7-10,10120,0.03,0.17,0.18,13.9,page_two_climber,review_title_meta_and_onpage_relevance,0.941,"ctr_far_below_band_norm, large_absolute_click_..."
9,10,A - review first,client 01,5-7,12215,0.05,0.23,0.22,21.9,page_two_climber,review_title_meta_and_onpage_relevance,0.938,"ctr_far_below_band_norm, large_absolute_click_..."


Top 200: 10 clients represented, largest holds 78% (that client is 40% of the universe).
That concentration is a standing review rule, not a footnote -- see Section 3.


### The decay / refresh insight

A queue is a photograph of a moving thing. The dataset has two consecutive 30-day windows, so *how fast this queue goes out of date* is measurable rather than guessable — and it turns out to be the single most load-bearing operational number in the playbook.

In [8]:
# ---- How fast does the queue go stale? -----------------------------------------------------
# 1. Does a flagged shortfall persist? 2. How much does the "right" top-K change in 30 days?
# 3. How noisy is a single window? 4. Is there an age/freshness gradient at all?
q["true_score_later"] = np.where(
    (q.exp_clicks_last >= MIN_EXP_CLICKS_30) & (q.ctr_last < SHORTFALL * q.norm_last),
    q.exp_clicks_last - q.clicks_last_30d, 0.0)

persist_flagged = float(q.loc[q.rule_flagged, "persisted_shortfall"].mean())
persist_unflagged = float(q.loc[~q.rule_flagged, "persisted_shortfall"].mean())

overlap = {}
for K in (50, 100, 200):
    a = set(np.argsort(-q.baseline_score.to_numpy(), kind="stable")[:K])
    b = set(np.argsort(-q.true_score_later.to_numpy(), kind="stable")[:K])
    overlap[K] = len(a & b) / K

ctr_move = float(((q.ctr_last - q.ctr_prev).abs() / q.ctr_prev.replace(0, np.nan) > 0.5).mean())
imp_move = float(((q.impressions_last_30d - q.impressions_prev_30d).abs() / q.impressions_prev_30d).median())

print(f"1. Persistence   : of pages the rule flagged in the earlier window, {persist_flagged:.1%} still had the")
print(f"                   shortfall 30 days later (n={int(q.rule_flagged.sum()):,}); of pages it did not flag, "
      f"{persist_unflagged:.1%} (n={int((~q.rule_flagged).sum()):,}).")
print(f"                   -> the flag carries real signal ({persist_flagged/persist_unflagged:.1f}x), and about half of a")
print("                   flagged queue resolves or stops being measurable within one window, untouched.")
print(f"2. Queue turnover: a top-K built on the earlier window overlaps the later window's 'correct' top-K by")
print("                   " + " | ".join(f"K={k}: {v:.0%}" for k, v in overlap.items()))
print("                   -> roughly half the list is different a month later.")
print(f"3. Single-window noise: {ctr_move:.0%} of pages moved CTR by more than 50% relative between the two")
print(f"                   windows; median absolute change in impressions was {imp_move:.0%}.")
print(f"4. Survivorship  : {DROPPED_LATER:,} pages eligible in the earlier window fell below the volume floor in")
print(f"                   the later one ({DROPPED_LATER / (len(q) + DROPPED_LATER):.0%}) and are invisible to every number above.")

1. Persistence   : of pages the rule flagged in the earlier window, 51.5% still had the
                   shortfall 30 days later (n=872); of pages it did not flag, 5.4% (n=8,964).
                   -> the flag carries real signal (9.5x), and about half of a
                   flagged queue resolves or stops being measurable within one window, untouched.
2. Queue turnover: a top-K built on the earlier window overlaps the later window's 'correct' top-K by
                   K=50: 54% | K=100: 54% | K=200: 52%
                   -> roughly half the list is different a month later.
3. Single-window noise: 41% of pages moved CTR by more than 50% relative between the two
                   windows; median absolute change in impressions was 32%.
4. Survivorship  : 1,683 pages eligible in the earlier window fell below the volume floor in
                   the later one (15%) and are invisible to every number above.


In [9]:
# ---- Is there a content-age or update-recency gradient? (the expected story) ----------------
q["age_bucket"] = pd.qcut(q.content_age_days, 5, duplicates="drop")
age_tbl = (q.groupby("age_bucket", observed=True)
           .agg(pages=("persisted_shortfall", "size"),
                shortfall_rate=("persisted_shortfall", "mean"),
                median_ctr_ratio=("ctr_ratio", "median")))
print("Persisted-shortfall rate by content age (quintiles of content_age_days):")
display(age_tbl.round(3))

upd = (q.groupby(pd.cut(q.days_since_last_update, [-1, 30, 90, 180, 365, 10**6],
                        labels=["0-30", "31-90", "91-180", "181-365", "365+"]), observed=True)
       .agg(pages=("persisted_shortfall", "size"), shortfall_rate=("persisted_shortfall", "mean")))
print("\nPersisted-shortfall rate by days since last update:")
display(upd.round(3))
print(f"days_since_last_update in this slice: min {q.days_since_last_update.min():.0f}, "
      f"median {q.days_since_last_update.median():.0f}, max {q.days_since_last_update.max():.0f}")
print("\nNegative result, stated plainly: across content-age quintiles the shortfall rate moves within a few")
print("points and does not trend. And the update-recency column is close to degenerate here -- almost every")
print("page sits in one of two buckets, with single- and double-digit counts in the rest. This slice therefore")
print("CANNOT support a 'content decays and refreshing fixes it' claim in either direction. Any such number")
print("from these cells would be a ratio from a tiny bucket, which writing-honest-claims says to drop, not dress up.")

Persisted-shortfall rate by content age (quintiles of content_age_days):


,pages,shortfall_rate,median_ctr_ratio
age_bucket,,,
"(89.999, 119.0]",1989,0.061,1.412
"(119.0, 167.0]",1951,0.100,0.986
"(167.0, 309.0]",1988,0.090,1.293
"(309.0, 421.0]",2003,0.118,0.979
"(421.0, 557.0]",1905,0.107,0.909



Persisted-shortfall rate by days since last update:


,pages,shortfall_rate
days_since_last_update,,
0-30,6227,0.089
31-90,29,0.241
91-180,3573,0.106
181-365,7,0.000


days_since_last_update in this slice: min 5, median 22, max 301

Negative result, stated plainly: across content-age quintiles the shortfall rate moves within a few
points and does not trend. And the update-recency column is close to degenerate here -- almost every
page sits in one of two buckets, with single- and double-digit counts in the rest. This slice therefore
CANNOT support a 'content decays and refreshing fixes it' claim in either direction. Any such number
from these cells would be a ratio from a tiny bucket, which writing-honest-claims says to drop, not dress up.


**What the decay numbers mean for the playbook — three operational rules, each traced to a measured number above:**

1. **The queue has a shelf life of about one month.** A top-K chosen on one window overlaps the next window's correct top-K by roughly half. So: **rebuild the queue every 30 days, and treat any queue older than 45 days as void.** Not because staleness is a theoretical risk — because half of it measurably stops being the right list.

2. **About half of a flagged queue resolves without anyone touching it.** Flagged pages persisted at roughly nine times the rate of unflagged ones, so the flag is carrying real signal — but a coin-flip's worth of the list would have looked "fixed" a month later with zero work done. **This is the number that makes a before/after reading of any refresh work worthless**, and it is the strongest argument in this notebook for holding out a control set before any of this is called a result.

3. **One window of CTR is a noisy read on a single page.** Two in five pages moved CTR by more than half, relative, between consecutive windows. That is why the volume floor and the expected-click gate stay in, why tier C exists, and why `low_volume_noisy_signal` is a reason code rather than a silent filter.

And the finding that did not appear: **no content-age or update-recency gradient is observable in this slice.** The portfolio's `days_since_last_update` is bimodal with near-empty middle buckets, so the refresh question this lane is named after is, on this data, *unanswerable rather than answered*. Recorded as a data-collection requirement for the paper's future-work section, not as a null effect.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended user.** One FlyRank content reviewer, internally, with roughly 50 review slots a week.

**Intended use.** Deciding **the order in which pages get a human look**, inside the portfolio and the window the queue was built from. That is the whole of it. Every claim this artefact is allowed to make is of the form *"on this data, pages ranked here had a measurably higher rate of a shortfall that persisted a month later."*

**The decision it replaces.** Not "should we rewrite this page" — nobody was doing that automatically. It replaces *"which of 9,836 eligible pages does a human open on Monday"*, which was previously answered by client priority and whoever asked most recently. The comparison that matters is against **the Week-4 rule** and against **random order**, both measured in Section 0.

**Out of scope, explicitly:**

- **It is not a content-quality score.** Nothing about writing quality, accuracy or usefulness is in the feature set.
- **It is not a forecast.** It does not say what CTR will be, or that anything will improve.
- **It is not causal, and cannot be made causal by rewording.** No intervention was assigned; no control group exists. The label records that a shortfall *persisted*, never that work *would have* fixed it.
- **It is not a client-facing artefact.** No number here is defensible in a client report, for the reasons in the limits register below.
- **It is not a writer or agency performance measure.** Using it that way would make it useless as a queue within one quarter — people optimise what they are scored on, and the score is a CTR ratio.
- **It is not a model of Google's ranking.** It models one portfolio's click behaviour relative to its own position-band norms.

**Where it stops being valid** — the boundary conditions, each one measured rather than asserted:

- **Outside the volume floor.** A page under ~167 impressions in 30 days was never scored. The score does not extend there; it is absent, not low.
- **Outside positions 1–20.** Band norms only exist for visible pages.
- **On navigational/branded intent.** The norms are built from a portfolio dominated by non-branded behaviour; branded queries behave differently. These rows are labelled `no_action`.
- **On a new client.** The client-grouped split is exactly what makes a new-client claim *possible* rather than certain — the model was scored on held-out clients, and the fold-to-fold spread on that test is wide (sd ≈ 0.15). Directionally it transfers; a point estimate for a new client does not.
- **After about 30 days.** See the decay section.

In [10]:
# ---- The limits register: every limit with the number that measures it ----------------------
largest_share = float(q.client_label.value_counts().iloc[0] / len(q))
largest_share_top = float(top.client_label.value_counts().iloc[0] / QUEUE_K)
spc_missing = float(q.sessions_per_click.isna().mean())
cpc_zero = float((q.cpc == 0).mean())
sv_zero = float((q.search_volume == 0).mean())

limits = pd.DataFrame([
    ("Client concentration",
     f"largest client = {largest_share:.0%} of the universe, {largest_share_top:.0%} of the top {QUEUE_K}",
     "the queue largely describes one client's pages; a portfolio-level claim is not supported"),
    ("Survivorship",
     f"{DROPPED_LATER:,} pages ({DROPPED_LATER/(len(q)+DROPPED_LATER):.0%}) dropped below the volume floor in the later window",
     "pages that collapsed in traffic are excluded from every rate above, in both directions"),
    ("Proxy label",
     "persisted_shortfall is a defined rule, not a reviewed outcome",
     "measures that a gap stayed open; says nothing about whether it was fixable"),
    ("Declared contamination",
     "avg_position is a 90-day average overlapping the outcome window",
     "Week 5 refit without position: PR-AUC moved ~0.03 -- bounded, not the source of the result"),
    ("Fold-to-fold spread",
     f"per-fold P@50 sd = {P50_FOLD_SD:.3f} around a mean of {P50_FOLD_MEAN:.3f}",
     "the pooled model-vs-rule gap sits inside this spread (Week 6); claim 'not worse, plausibly better'"),
    ("Single time window",
     "one pair of consecutive 30-day windows",
     "no seasonality, no algorithm-update period, no repeat measurement"),
    ("Unreliable columns",
     f"cpc = 0 on {cpc_zero:.0%} of rows; search_volume = 0 on {sv_zero:.0%}; sessions_per_click missing on {spc_missing:.0%}",
     "no monetary value is computed anywhere in this playbook -- the inputs would not carry it"),
    ("Sample, not the release",
     f"{len(df):,}-row anonymized slice",
     "the full ~79M-row release was not used; numbers are of this slice"),
], columns=["limit", "measured as", "what it forbids"])
pd.set_option("display.max_colwidth", 90)
display(limits)
print("Rule of thumb this table encodes: a limit without a number attached is a disclaimer, and")
print("disclaimers get skipped. Each row above can be checked against a cell in this notebook.")

,limit,measured as,what it forbids
0,Client concentration,"largest client = 40% of the universe, 78% of the top 200",the queue largely describes one client's pages; a portfolio-level claim is not supported
1,Survivorship,"1,683 pages (15%) dropped below the volume floor in the later window","pages that collapsed in traffic are excluded from every rate above, in both directions"
2,Proxy label,"persisted_shortfall is a defined rule, not a reviewed outcome",measures that a gap stayed open; says nothing about whether it was fixable
3,Declared contamination,avg_position is a 90-day average overlapping the outcome window,"Week 5 refit without position: PR-AUC moved ~0.03 -- bounded, not the source of the re..."
4,Fold-to-fold spread,per-fold P@50 sd = 0.151 around a mean of 0.648,"the pooled model-vs-rule gap sits inside this spread (Week 6); claim 'not worse, plaus..."
5,Single time window,one pair of consecutive 30-day windows,"no seasonality, no algorithm-update period, no repeat measurement"
6,Unreliable columns,cpc = 0 on 76% of rows; search_volume = 0 on 37%; sessions_per_click missing on 20%,no monetary value is computed anywhere in this playbook -- the inputs would not carry it
7,"Sample, not the release","30,000-row anonymized slice",the full ~79M-row release was not used; numbers are of this slice


Rule of thumb this table encodes: a limit without a number attached is a disclaimer, and
disclaimers get skipped. Each row above can be checked against a cell in this notebook.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**The queue's output is a review appointment, not an instruction.** Every row arrives with mandatory checks attached, and some rows cannot be worked at all until a prior question is answered.

### Mandatory checks, per row

| Flag | Fires when | The reviewer must |
|---|---|---|
| `CHECK_ANALYTICS_FIRST` | GA4 sessions per GSC click outside 0.5–2.0, or missing | confirm the page's tracking before treating the gap as a content problem — a measurement artefact and a content problem look identical in this data |
| `CHECK_SITE_WIDE_CAUSE` | the page's client holds >40% of the top 200 | ask whether the cause is a template, a sitewide title pattern or a tracking change **before** opening page-by-page work |
| `CHECK_INTENT_MATCH` | commercial/transactional intent on a generic article format | judge whether the page is the wrong *format* for the query; if it is, a title rewrite is the wrong action |
| `CHECK_EVIDENCE_THIN` | tier C, or impressions below 400 in the window | treat as monitor-only; the signal is inside the noise measured in the decay section |
| `CHECK_BRANDED_QUERY` | navigational intent | remove from the queue — band norms do not describe branded behaviour |
| `CHECK_TOPIC_SENSITIVITY` | **always, every row** | confirm the page is not medical, financial, legal or otherwise regulated. **Topic is not in the feature set at all**, so the queue is structurally blind to this and cannot flag it |

The last one is deliberate and is the most important row in the table. The honest statement is not "the model handles sensitive topics carefully" — it is *"the model cannot see them, so a human is the only control that exists."*

### The no-go list — what must never be automated

1. **Auto-publishing anything.** No title, meta description or body text goes live from this queue without a human writing and approving it. The model has no evidence about what a *good* replacement is; it only ranks where to look.
2. **Acting on navigational/branded pages.** Excluded by design, not by discretion.
3. **De-indexing, deleting, consolidating or redirecting.** Irreversible actions on the strength of a 30-day CTR ratio, from a model whose false positives are *self-repairing pages*, is the worst available trade.
4. **Acting while `CHECK_ANALYTICS_FIRST` is open.** The flag fires on about half the pages in this slice; rewriting a page whose tracking is broken produces a change in nothing and a confident story about why it worked.
5. **Page-by-page work on a dominated queue.** When most of the list is one client, page-level work is the expensive way to fix what may be one template.
6. **Scoring pages below the volume floor.** Absence of a score is not a low score; do not backfill it with a guess.
7. **Any client-facing number, forecast or projected uplift.** Not "phrase it carefully" — do not produce it. There is no design behind it.
8. **Using the score to evaluate a writer, agency or team.**
9. **Running a queue older than 45 days.** Measured, in the decay section: roughly half of it is the wrong list by then.
10. **Reading before/after on reviewed pages as an effect.** About half of flagged pages resolved untouched, so a naive before/after would show "success" on a coin flip. **Any claim that this work helped requires a held-out control set decided in advance.**

### One thing to automate, gladly

Rebuilding the queue itself — the notebook run, the exports, the monitoring checks in Section 4. **Automate the measuring; never automate the acting.**

In [11]:
# ---- Human-review flags, attached per row ---------------------------------------------------
CONCENTRATION_LIMIT = 0.40
dominant = set(top.client_label.value_counts()[lambda s: s / QUEUE_K > CONCENTRATION_LIMIT].index)

def review_flags(r):
    f = []
    if pd.isna(r.sessions_per_click) or not (SPC_LO <= r.sessions_per_click <= SPC_HI):
        f.append("CHECK_ANALYTICS_FIRST")
    if r.client_label in dominant:
        f.append("CHECK_SITE_WIDE_CAUSE")
    if r.main_intent in ("transactional", "commercial") and r.content_type == "keyword article":
        f.append("CHECK_INTENT_MATCH")
    if r.tier == "C - monitor only" or r.impressions_prev_30d < LOW_IMPRESSIONS:
        f.append("CHECK_EVIDENCE_THIN")
    if r.main_intent == "navigational":
        f.append("CHECK_BRANDED_QUERY")
    f.append("CHECK_TOPIC_SENSITIVITY")   # always: topic is not in the feature set
    return f

q["review_flags"] = [review_flags(r) for r in q.itertuples()]
q["blocked_until_checked"] = q.review_flags.map(
    lambda fs: any(x in fs for x in ("CHECK_ANALYTICS_FIRST", "CHECK_BRANDED_QUERY")))
top = q[q.in_top_200]

flag_counts = Counter(f for fs in top.review_flags for f in fs)
print(f"Mandatory checks on the working queue (top {QUEUE_K}):")
display(pd.DataFrame(sorted(flag_counts.items(), key=lambda kv: -kv[1]), columns=["flag", "pages"])
        .assign(share=lambda t: (t.pages / QUEUE_K).round(3)))
print(f"{int(top.blocked_until_checked.sum())} of the top {QUEUE_K} are BLOCKED from content work until a prior")
print("question is answered (tracking, or branded intent). They keep their rank -- they are not silently")
print("dropped, because a reviewer needs to see that the top of the list is waiting on something.")
print(f"\nAnd every row carries CHECK_TOPIC_SENSITIVITY, because topic is not among the {len(FEATURES)} features.")

Mandatory checks on the working queue (top 200):


,flag,pages,share
0,CHECK_TOPIC_SENSITIVITY,200,1.000
1,CHECK_SITE_WIDE_CAUSE,155,0.775
2,CHECK_ANALYTICS_FIRST,127,0.635
3,CHECK_INTENT_MATCH,73,0.365


127 of the top 200 are BLOCKED from content work until a prior
question is answered (tracking, or branded intent). They keep their rank -- they are not silently
dropped, because a reviewer needs to see that the top of the list is waiting on something.

And every row carries CHECK_TOPIC_SENSITIVITY, because topic is not among the 16 features.


### Cost and value — what a reviewer week actually buys

One side of this is measured and one side is not, and the playbook is only useful if it says which is which.

In [12]:
# ---- Cost is measurable. Value is not. Say so with numbers. ---------------------------------
SLOTS = 50
t50 = q.head(SLOTS)
gap_sum = float(t50.click_gap_30d.sum())
gap_med = float(t50.click_gap_30d.median())

print("MEASURED -- where 50 slots land (out-of-fold, this run):")
for label, p in [("random order", BASE_RATE), ("Week-4 rule", P50_RULE), ("model queue (pooled)", P50_MODEL),
                 ("model queue (per-fold mean)", P50_FOLD_MEAN)]:
    print(f"  {label:<28} {p*SLOTS:>5.1f} of {SLOTS} slots land on a page whose shortfall persisted  (P@{SLOTS} {p:.3f})")
print(f"\n  vs random order, the model queue avoids ~{(P50_MODEL-BASE_RATE)*SLOTS:.0f} wasted slots per reviewer-week;")
print(f"  vs the Week-4 rule, ~{(P50_MODEL-P50_RULE)*SLOTS:.0f} -- and Week 6 showed that second number is inside")
print(f"  the fold-to-fold spread (sd {P50_FOLD_SD:.3f}), so it is directional at best.")

print(f"\nMEASURED -- what is at stake in those 50 pages: {gap_sum:,.0f} clicks of shortfall per 30 days")
print(f"  (median {gap_med:.0f} clicks per page), where 'shortfall' = expected clicks at the page's band norm,")
print("  minus actual. That is a measured gap, not a recoverable amount.")

print("\nNOT MEASURED, and not estimated here:")
print("  - how much of that gap a review can recover: no intervention was run, so the answer is unknown")
print(f"  - what a click is worth: cpc is 0 on {cpc_zero:.0%} of rows and search_volume on {sv_zero:.0%},")
print("    so no monetary figure is computed anywhere in this playbook -- the inputs cannot carry one")
print("  - reviewer hours per page: not in the data; FlyRank has it and should supply it")

print("\nThe honest framing for the paper is a BREAK-EVEN, stated as a condition and not a projection:")
print(f"  a reviewer week costs {SLOTS} slots. It is put in front of {gap_sum:,.0f} clicks/30d of measured shortfall,")
print(f"  of which ~{P50_MODEL*SLOTS:.0f} of 50 pages still had that gap open a month later. Whether the work pays")
print("  depends entirely on the recovery rate -- which THIS DESIGN CANNOT MEASURE and the decay section shows")
print("  a naive before/after would get wrong, since ~half of flagged pages resolved untouched.")
print("  Measuring it needs a held-out control set. That is the single highest-value next experiment.")

MEASURED -- where 50 slots land (out-of-fold, this run):
  random order                   4.8 of 50 slots land on a page whose shortfall persisted  (P@50 0.095)
  Week-4 rule                   44.0 of 50 slots land on a page whose shortfall persisted  (P@50 0.880)
  model queue (pooled)          46.0 of 50 slots land on a page whose shortfall persisted  (P@50 0.920)
  model queue (per-fold mean)   32.4 of 50 slots land on a page whose shortfall persisted  (P@50 0.648)

  vs random order, the model queue avoids ~41 wasted slots per reviewer-week;
  vs the Week-4 rule, ~2 -- and Week 6 showed that second number is inside
  the fold-to-fold spread (sd 0.151), so it is directional at best.

MEASURED -- what is at stake in those 50 pages: 1,403 clicks of shortfall per 30 days
  (median 22 clicks per page), where 'shortfall' = expected clicks at the page's band norm,
  minus actual. That is a measured gap, not a recoverable amount.

NOT MEASURED, and not estimated here:
  - how much of that 

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring here is a **monthly checklist a person runs**, not a service. Every check is computed from the same two things the notebook already needs (the export, and next month's data), and each one has a number attached now so that next month's run has something to compare against.

The thresholds are set from **this run's observed values**, with amber at roughly a 30% relative move and red at 50%, except where a check has a natural hard edge. They are starting positions to be revised once a second month exists — and that is stated rather than hidden, because a threshold invented from one observation is a hypothesis.

**Retrain triggers — any one of these, and the model is refit before the queue is trusted again:**

- any monitoring check red, or the same check amber two months running
- a new client joins the portfolio at more than ~10% of eligible pages (the grouped split's transfer claim was never tested on them)
- band norms move more than 20% — the label is defined against those norms, so they moving means the *label* changed, not just the inputs
- a schema or definition change in any input column
- more than 90 days since the last fit, regardless of whether anything looks wrong

**Retraining is not a fix for a broken label.** If the band norms shift, refitting on the new norms produces a confident model of a different question. That case needs the label revisited first, which is a human decision.

In [13]:
# ---- Monitoring spec, with this run's values as the baseline --------------------------------
band_norms_now = q.groupby("band", observed=True).norm_prev.first().round(2).to_dict()
queue_overlap_now = overlap[200]

monitor = pd.DataFrame([
    ("base rate of persisted_shortfall", f"{BASE_RATE:.3f}", "±30%", "±50%", "monthly",
     "the thing being ranked changed; precision@K is no longer comparable month to month"),
    ("P@50 of the shipped queue", f"{P50_MODEL:.3f}", f"< {P50_RULE:.2f} (the rule)", f"< {2*BASE_RATE:.2f}", "monthly",
     "if it falls below the Week-4 rule, ship the rule -- it is free and needs no fitting"),
    ("largest client share of top 200", f"{largest_share_top:.2f}", "> 0.80", "> 0.90", "monthly",
     "the queue has become one client's worklist; portfolio framing no longer honest"),
    ("month-over-month top-200 overlap", f"{queue_overlap_now:.2f}", "< 0.35 or > 0.85", "< 0.20", "monthly",
     "too low = churn/noise; too high = the queue is frozen and nothing is being worked"),
    ("band norm drift (median %CTR/band)", str(band_norms_now), "any band ±20%", "any band ±35%", "monthly",
     "the LABEL is defined against these -- drift here changes the question, not just the answer"),
    ("sessions_per_click missing share", f"{spc_missing:.2f}", "> 0.30", "> 0.45", "monthly",
     "an input is degrading; review flags start firing on noise"),
    ("pages dropping below volume floor", f"{DROPPED_LATER/(len(q)+DROPPED_LATER):.2f}", "> 0.25", "> 0.35", "monthly",
     "survivorship is growing; rates in the playbook describe an increasingly filtered set"),
    ("reviewer accept rate of queue rows", "NOT YET INSTRUMENTED", "< 0.50", "< 0.30", "monthly",
     "the only check that tests the queue against human judgment -- must be captured from week one"),
], columns=["check", "this run", "amber", "red", "cadence", "what it would mean"])
display(monitor)
print("The last row has no baseline because nobody has worked this queue yet. It is listed first in")
print("priority for exactly that reason: every other check tests the pipeline against itself, and only")
print("reviewer feedback tests it against the world. Instrumenting it is a prerequisite, not an extra.")

,check,this run,amber,red,cadence,what it would mean
0,base rate of persisted_shortfall,0.095,±30%,±50%,monthly,the thing being ranked changed; precision@K is no longer comparable month to month
1,P@50 of the shipped queue,0.920,< 0.88 (the rule),< 0.19,monthly,"if it falls below the Week-4 rule, ship the rule -- it is free and needs no fitting"
2,largest client share of top 200,0.78,> 0.80,> 0.90,monthly,the queue has become one client's worklist; portfolio framing no longer honest
3,month-over-month top-200 overlap,0.53,< 0.35 or > 0.85,< 0.20,monthly,too low = churn/noise; too high = the queue is frozen and nothing is being worked
4,band norm drift (median %CTR/band),"{'0-3': 0.21, '3-5': 0.32, '5-7': 0.23, '7-10': 0.17, '10-15': 0.14, '15-20': 0.12}",any band ±20%,any band ±35%,monthly,"the LABEL is defined against these -- drift here changes the question, not just the an..."
5,sessions_per_click missing share,0.20,> 0.30,> 0.45,monthly,an input is degrading; review flags start firing on noise
6,pages dropping below volume floor,0.15,> 0.25,> 0.35,monthly,survivorship is growing; rates in the playbook describe an increasingly filtered set
7,reviewer accept rate of queue rows,NOT YET INSTRUMENTED,< 0.50,< 0.30,monthly,the only check that tests the queue against human judgment -- must be captured from we...


The last row has no baseline because nobody has worked this queue yet. It is listed first in
priority for exactly that reason: every other check tests the pipeline against itself, and only
reviewer feedback tests it against the world. Instrumenting it is a prerequisite, not an extra.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three kinds of artefact, on purpose:

- **`work/outputs/w07_action_queue.csv`** — the full ranked queue with tiers, archetypes, actions, reason codes and review flags. **Not committed:** the CI leak-guard blocks data files under `work/`, and this notebook regenerates it deterministically (seed 42).
- **`work/outputs/w07_playbook_metrics.json`** — committed. Every number quoted in the paper's recommendations section traces back to a key in this file.
- **`work/figures/*.png`** — committed, and sized for the paper.

In [14]:
# ---- Figures ---------------------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 140, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
FIGS = {}

# (1) Precision@K -- the queue's core claim, with the reviewer's K marked
ks = [10, 20, 30, 50, 75, 100, 150, 200, 300, 500]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(ks, [precision_at_k(q.p_model, q.persisted_shortfall, k) for k in ks], "o-", label="model queue (out-of-fold)")
ax.plot(ks, [precision_at_k(q.baseline_score, q.persisted_shortfall, k) for k in ks], "s--", label="Week-4 rule")
ax.axhline(BASE_RATE, color="gray", ls=":", label=f"base rate ({BASE_RATE:.3f})")
ax.axvline(50, color="k", lw=0.8, alpha=0.4)
ax.annotate("a reviewer's week\n(K=50)", (50, 0.30), fontsize=8, ha="left", xytext=(58, 0.28))
ax.set(xlabel="K (pages opened, in rank order)", ylabel="precision@K", ylim=(0, 1.02),
       title="Precision@K: how many opened pages had a shortfall that persisted")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); FIGS["w07_precision_at_k.png"] = fig

# (2) The decay picture -- the playbook's most load-bearing operational number
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.2))
axes[0].bar(["flagged\n(earlier window)", "not flagged"], [persist_flagged, persist_unflagged],
            color=["#c44", "#bbb"])
for i, v in enumerate([persist_flagged, persist_unflagged]):
    axes[0].text(i, v + 0.015, f"{v:.1%}", ha="center", fontsize=9)
axes[0].set(ylabel="share with shortfall still open 30d later", ylim=(0, 0.65),
            title="Persistence of a flagged shortfall")
axes[1].bar([str(k) for k in overlap], list(overlap.values()), color="#46a")
for i, v in enumerate(overlap.values()):
    axes[1].text(i, v + 0.015, f"{v:.0%}", ha="center", fontsize=9)
axes[1].set(xlabel="K", ylabel="overlap with next window's correct top-K", ylim=(0, 1.0),
            title="How much of the queue is still right\nafter 30 days")
fig.tight_layout(); FIGS["w07_queue_decay.png"] = fig

# (3) Tier calibration -- what a tier label is allowed to promise
fig, ax = plt.subplots(figsize=(5.6, 3.2))
tr = tiers.reset_index()
ax.barh(tr.tier, tr.observed_shortfall_rate, color=["#2a7", "#fa3", "#bbb"])
for i, (v, n) in enumerate(zip(tr.observed_shortfall_rate, tr.pages)):
    ax.text(v + 0.012, i, f"{v:.1%}  (n={n:,})", va="center", fontsize=8)
ax.axvline(BASE_RATE, color="k", ls=":", lw=1)
ax.text(BASE_RATE + 0.005, -0.45, f"base rate {BASE_RATE:.1%}", fontsize=8)
ax.set(xlabel="observed rate of persisted shortfall", xlim=(0, 0.68),
       title="What each tier label is allowed to promise")
fig.tight_layout(); FIGS["w07_tier_calibration.png"] = fig

# (4) What the working queue is made of
fig, ax = plt.subplots(figsize=(6.4, 3.2))
m = mix.sort_values("top_200")[lambda t: t.top_200 > 0]
ax.barh(m.index, m.top_200, color="#46a")
for i, (n, h) in enumerate(zip(m.top_200, m.top_200_hit_rate)):
    ax.text(n + 2, i, f"n={n}" + (f", hit {h:.0%}" if n >= 30 else ", n too small to quote"), va="center", fontsize=8)
ax.set(xlabel=f"pages in the top {QUEUE_K}", xlim=(0, m.top_200.max() * 1.6),
       title=f"Archetype mix of the working queue (top {QUEUE_K})")
fig.tight_layout(); FIGS["w07_archetype_mix.png"] = fig

for name, f in FIGS.items():
    f.savefig(FIG / name, bbox_inches="tight")
    plt.close(f)
print("figures written to work/figures/:")
for name in FIGS:
    print(f"  {name}  ({(FIG / name).stat().st_size / 1024:.0f} KB)")

figures written to work/figures/:
  w07_precision_at_k.png  (49 KB)
  w07_queue_decay.png  (51 KB)
  w07_tier_calibration.png  (34 KB)
  w07_archetype_mix.png  (36 KB)


In [15]:
# ---- The queue export -- the file next week's paper builds on -------------------------------
export = q[[
    "queue_rank", "in_top_200", "tier", "p_model", "content_id", "client_label",
    "archetype", "suggested_action", "action_priority",
    "band", "avg_position", "impressions_prev_30d", "clicks_prev_30d",
    "ctr_prev", "norm_prev", "ctr_ratio", "click_gap_30d",
    "word_count", "main_intent", "content_type", "days_since_last_update",
    "sessions_per_click", "rule_flagged", "baseline_score",
    "blocked_until_checked", "persisted_shortfall",
]].copy()
export["reason_codes"] = q.reason_codes.map("|".join)
export["review_flags"] = q.review_flags.map("|".join)
export = export.round({"p_model": 4, "ctr_prev": 3, "norm_prev": 3, "ctr_ratio": 3,
                       "click_gap_30d": 1, "sessions_per_click": 3, "baseline_score": 2})

# persisted_shortfall is the OUTCOME. It is exported for the paper's evaluation tables only --
# it is not knowable at decision time and must never be shown to a reviewer working the queue.
queue_path = OUT / "w07_action_queue.csv"
export.to_csv(queue_path, index=False)
print(f"{queue_path.relative_to(root)}  --  {len(export):,} rows x {export.shape[1]} columns "
      f"({queue_path.stat().st_size/1024:.0f} KB)")
print("   gitignored by design (work/**/*.csv); regenerated deterministically by this notebook, seed 42")
print("   client_id replaced by an ordinal label; content_id is the source file's anonymized hash")
display(export.head(4))

work\outputs\w07_action_queue.csv  --  9,836 rows x 28 columns (3502 KB)
   gitignored by design (work/**/*.csv); regenerated deterministically by this notebook, seed 42
   client_id replaced by an ordinal label; content_id is the source file's anonymized hash


,queue_rank,in_top_200,tier,p_model,content_id,client_label,archetype,suggested_action,action_priority,band,...,main_intent,content_type,days_since_last_update,sessions_per_click,rule_flagged,baseline_score,blocked_until_checked,persisted_shortfall,reason_codes,review_flags
0,1,True,A - review first,0.9579,content_896bf2cc27b7,client 01,page_one_snippet_gap,review_title_and_meta,high,3-5,...,informational,keyword article,104,2.533,True,88.90,True,1,ctr_far_below_band_norm|large_absolute_click_gap|high_impression_volume|page_one_posit...,CHECK_ANALYTICS_FIRST|CHECK_SITE_WIDE_CAUSE|CHECK_TOPIC_SENSITIVITY
1,2,True,A - review first,0.9577,content_7da14d2209fb,client 01,page_two_climber,review_title_meta_and_onpage_relevance,high,5-7,...,informational,keyword article,104,5.000,True,16.67,True,1,ctr_far_below_band_norm|large_absolute_click_gap|high_impression_volume|page_two_posit...,CHECK_ANALYTICS_FIRST|CHECK_SITE_WIDE_CAUSE|CHECK_TOPIC_SENSITIVITY
2,3,True,A - review first,0.9556,content_2e0b3dc70916,client 02,page_two_climber,review_title_meta_and_onpage_relevance,high,5-7,...,informational,keyword article,104,1.000,True,19.67,False,1,ctr_far_below_band_norm|large_absolute_click_gap|high_impression_volume|page_two_posit...,CHECK_TOPIC_SENSITIVITY
3,4,True,A - review first,0.9507,content_609902f4bc8c,client 02,page_two_climber,review_title_meta_and_onpage_relevance,high,5-7,...,informational,keyword article,34,0.750,True,28.91,False,1,ctr_far_below_band_norm|large_absolute_click_gap|high_impression_volume|page_two_position,CHECK_TOPIC_SENSITIVITY


In [16]:
# ---- The receipts: every number the paper's recommendations section will quote ---------------
metrics = {
    "notebook": "w07_action_playbook.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "seed": SEED,
    "sklearn": sklearn.__version__,
    "intended_use": {
        "user": "one internal FlyRank content reviewer, ~50 review slots per week",
        "decision": "order in which pages get a human look",
        "replaces": "client priority / most recent request",
        "not_for": ["client-facing reporting", "forecasting", "causal claims",
                     "content quality scoring", "writer or agency evaluation",
                     "any automated publishing action"],
        "production": False,
    },
    "universe": {
        "pages": int(len(q)), "clients": int(q.client_id.nunique()),
        "base_rate": round(BASE_RATE, 4),
        "dropped_not_eligible_later_window": DROPPED_LATER,
        "largest_client_share_of_universe": round(largest_share, 3),
        "largest_client_share_of_top200": round(largest_share_top, 3),
    },
    "queue": {
        "scored_rows": int(len(q)), "working_queue_K": QUEUE_K,
        "tiers": {str(k): {"pages": int(v.pages), "observed_shortfall_rate": round(float(v.observed_shortfall_rate), 4),
                            "lift_vs_base": round(float(v.lift_vs_base), 2)}
                   for k, v in tiers.iterrows()},
        "archetype_mix_top200": {k: int(v) for k, v in top.archetype.value_counts().items()},
        "archetype_action_map": {k: {"action": v[0], "priority": v[1]} for k, v in ACTION.items()},
        "blocked_until_checked_top200": int(top.blocked_until_checked.sum()),
        "reason_code_frequency": {k: int(v) for k, v in sorted(freq.items(), key=lambda kv: -kv[1])},
    },
    "ranking_quality": {
        "P@50_model_pooled": round(P50_MODEL, 4),
        "P@50_rule_pooled": round(P50_RULE, 4),
        "P@50_base_rate": round(BASE_RATE, 4),
        "P@50_model_per_fold_mean": round(P50_FOLD_MEAN, 4),
        "P@50_model_per_fold_sd": round(P50_FOLD_SD, 4),
        "claim": ("not worse than the Week-4 rule, plausibly somewhat better; the pooled gap sits "
                  "inside the fold-to-fold spread (Week 6)"),
    },
    "decay_refresh": {
        "persistence_flagged": round(persist_flagged, 4),
        "persistence_unflagged": round(persist_unflagged, 4),
        "persistence_lift": round(persist_flagged / persist_unflagged, 2),
        "topK_overlap_with_next_window": {str(k): round(v, 3) for k, v in overlap.items()},
        "share_pages_ctr_moved_over_50pct": round(ctr_move, 3),
        "median_abs_impression_change": round(imp_move, 3),
        "content_age_gradient": "none observable across quintiles in this slice",
        "update_recency_gradient": "not testable -- days_since_last_update is bimodal with near-empty middle buckets",
        "operational_rules": {"rebuild_queue_days": 30, "queue_void_after_days": 45,
                               "before_after_readings_invalid_without_control_set": True},
    },
    "cost_value": {
        "slots_per_week": SLOTS,
        "slots_well_spent_random": round(BASE_RATE * SLOTS, 1),
        "slots_well_spent_rule": round(P50_RULE * SLOTS, 1),
        "slots_well_spent_model_pooled": round(P50_MODEL * SLOTS, 1),
        "slots_well_spent_model_per_fold": round(P50_FOLD_MEAN * SLOTS, 1),
        "click_shortfall_in_top50_per_30d": round(gap_sum, 1),
        "median_click_gap_top50": round(gap_med, 1),
        "monetary_value": None,
        "monetary_value_reason": f"cpc is 0 on {cpc_zero:.0%} of rows and search_volume on {sv_zero:.0%}",
        "recovery_rate": None,
        "recovery_rate_reason": "no intervention was assigned; requires a held-out control set",
    },
    "human_review": {
        "mandatory_flags": sorted(set(f for fs in q.review_flags for f in fs)),
        "always_on": "CHECK_TOPIC_SENSITIVITY -- topic is not in the feature set, so a human is the only control",
        "no_go": [
            "auto-publishing any title, meta or body text",
            "acting on navigational/branded pages",
            "de-indexing, deleting, consolidating or redirecting",
            "acting while the analytics-mismatch flag is open",
            "page-by-page work while one client dominates the queue",
            "scoring pages below the volume floor",
            "any client-facing number, forecast or projected uplift",
            "using the score to evaluate a writer, agency or team",
            "running a queue older than 45 days",
            "reading before/after on reviewed pages as an effect",
        ],
    },
    "monitoring": {
        "cadence": "monthly, run by a person",
        "checks": monitor.to_dict(orient="records"),
        "retrain_triggers": [
            "any check red, or the same check amber two months running",
            "new client above ~10% of eligible pages",
            "band norms move more than 20% (this changes the label, not just the inputs)",
            "schema or definition change in any input column",
            "more than 90 days since the last fit",
        ],
        "caveat": "thresholds are set from a single observation and are hypotheses until a second month exists",
    },
    "limits": limits.to_dict(orient="records"),
    "exports": {
        "queue_csv": "work/outputs/w07_action_queue.csv (gitignored, regenerated by this notebook)",
        "metrics_json": "work/outputs/w07_playbook_metrics.json (committed)",
        "figures": sorted(FIGS.keys()),
    },
}
metrics_path = OUT / "w07_playbook_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"{metrics_path.relative_to(root)} written ({metrics_path.stat().st_size/1024:.1f} KB)")
print(f"top-level keys: {list(metrics)}")

work\outputs\w07_playbook_metrics.json written (10.8 KB)
top-level keys: ['notebook', 'lane', 'seed', 'sklearn', 'intended_use', 'universe', 'queue', 'ranking_quality', 'decay_refresh', 'cost_value', 'human_review', 'monitoring', 'limits', 'exports']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [17]:
# ---- Assertions, so the self-check above is checked rather than asserted ---------------------
checks = {
    "every scored page has exactly one archetype and one action":
        q.archetype.notna().all() and q.suggested_action.notna().all() and len(q.archetype.unique()) <= len(ACTION),
    "every page carries at least one reason code":
        q.reason_codes.map(len).gt(0).all(),
    "every page carries the always-on topic-sensitivity check":
        q.review_flags.map(lambda fs: "CHECK_TOPIC_SENSITIVITY" in fs).all(),
    "tier labels are backed by an observed rate, and the tiers are ordered":
        tiers.observed_shortfall_rate.is_monotonic_decreasing,
    "the outcome column never drives the ranking (rank is a function of p_model alone)":
        q.queue_rank.equals(pd.Series(np.arange(1, len(q) + 1), name="queue_rank")) and
        q.p_model.is_monotonic_decreasing,
    "no monetary value is claimed anywhere in the exported metrics":
        metrics["cost_value"]["monetary_value"] is None and metrics["cost_value"]["recovery_rate"] is None,
    "navigational pages are routed to no_action":
        set(q.loc[q.main_intent == "navigational", "suggested_action"]) <= {"no_action_remove_from_queue"},
    "the no-go list is non-empty and exported":
        len(metrics["human_review"]["no_go"]) >= 8,
    "monitoring has a baseline value for every check that has one to have":
        len(metrics["monitoring"]["checks"]) >= 8,
    "queue CSV and metrics JSON both exist on disk":
        queue_path.exists() and metrics_path.exists(),
    "all four figures written":
        all((FIG / n).exists() for n in FIGS),
    "no client hash reaches the export (ordinal label only)":
        "client_id" not in export.columns and export.client_label.str.startswith("client ").all(),
}
for k, v in checks.items():
    print(f"[{'x' if v else ' '}] {k}")
assert all(checks.values()), [k for k, v in checks.items() if not v]
print(f"\nAll {len(checks)} checks pass. Queue: {len(export):,} rows, working K={QUEUE_K}, seed {SEED}.")

[x] every scored page has exactly one archetype and one action
[x] every page carries at least one reason code
[x] every page carries the always-on topic-sensitivity check
[x] tier labels are backed by an observed rate, and the tiers are ordered
[x] the outcome column never drives the ranking (rank is a function of p_model alone)
[x] no monetary value is claimed anywhere in the exported metrics
[x] navigational pages are routed to no_action
[x] the no-go list is non-empty and exported
[x] monitoring has a baseline value for every check that has one to have
[x] queue CSV and metrics JSON both exist on disk
[x] all four figures written
[x] no client hash reaches the export (ordinal label only)

All 12 checks pass. Queue: 9,836 rows, working K=200, seed 42.
Commit: work/notebooks/w07_action_playbook.ipynb, work/figures/*.png, work/outputs/w07_playbook_metrics.json
